<a href="https://colab.research.google.com/github/mkvkanpur/hpc_book/blob/main/PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

x = torch.tensor([1, 2, 3, 4])

print(x.shape)

y = x.view(4)

print(x.view(4), x.view(2,2))


torch.Size([4])
tensor([1, 2, 3, 4]) tensor([[1, 2],
        [3, 4]])


In [ ]:
f = torch.tensor([1.,2.,3.,4.,5.])
f_tensor = f.view(1,1,-1)
print(f_tensor)

tensor([[[1., 2., 3., 4., 5.]]])


In [ ]:
kernel = torch.tensor([-1., 0., 1.])
print(kernel.shape)
# torch.Size([3])

torch.Size([3])


In [ ]:
import torch
x = torch.randn(2,2)
y = x.t()
print(f"y = {y}")
print(y.is_contiguous())
z = y


y = tensor([[0.3548],
        [0.7918]])
True


In [ ]:
import torch

# 1. Create a contiguous 2x3 tensor
# Memory: [1, 2, 3, 4, 5, 6]
x = torch.tensor([[1, 2], [3, 4]])
print(f"Original x is contiguous: {x.is_contiguous()}") # True

# 2. Transpose it
# The GPU just changes the 'Stride'. It doesn't move the 1, 2, 3...
# Memory still looks like: [1, 2, 3, 4, 5, 6]
# But the 'View' is now 3x2.
y = x.t()
print(f"Transposed y is contiguous: {y.is_contiguous()}") # False

# 3. Try to use .view() on the non-contiguous tensor
try:
    z = y.view(-1)
except RuntimeError as e:
    print(f"Caught Error: {e}")
    # Error: view size is not compatible with input tensor's size and stride

y_fixed = y.contiguous()

print(f"Fixed y is contiguous: {y_fixed.is_contiguous()}") # True

# 5. Now .view() works perfectly
z = y_fixed.view(-1)
print(f"Flattened z: {z}")
# Result: [1, 4, 2, 5, 3, 6]

Original x is contiguous: True
Transposed y is contiguous: False
Caught Error: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
Fixed y is contiguous: True
Flattened z: tensor([1, 3, 2, 4])


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Initialize two fields (u and v) with 8 points each
x = torch.linspace(0, 7, 8, device=device)
u = 10.0 * x                # Linear: [0, 10, 20, 30, 40, 50, 60, 70]
v = x**2                   # Quadratic: [0, 1, 4, 9, 16, 25, 36, 49]

uv_tensor = torch.stack([u, v], dim=0)
print(uv_tensor)
uv_tensor = torch.stack([u, v], dim=1)
print(uv_tensor)

print(v.view(2,2,2))

tensor([[ 0., 10., 20., 30., 40., 50., 60., 70.],
        [ 0.,  1.,  4.,  9., 16., 25., 36., 49.]], device='cuda:0')
tensor([[ 0.,  0.],
        [10.,  1.],
        [20.,  4.],
        [30.,  9.],
        [40., 16.],
        [50., 25.],
        [60., 36.],
        [70., 49.]], device='cuda:0')
tensor([[[ 0.,  1.],
         [ 4.,  9.]],

        [[16., 25.],
         [36., 49.]]], device='cuda:0')


In [ ]:
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

# Channel 1
c1 = torch.tensor([1., 2., 3., 4., 5.], device=device)

# Reshape c1 to (batch_size, in_channels, length) for conv1d
c1_input = c1.view(1, 1, -1)

kernel = torch.tensor([-1.0, 0.0, 1.0], device=device)
# Reshape kernel to (out_channels, in_channels, kernel_width)
kernel_filter = kernel.view(1, 1, 3)

df = F.conv1d(c1_input, kernel_filter, padding=0)
print(df)

tensor([[[2., 2., 2.]]], device='cuda:0')


In [ ]:
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

# Channel 1
c1 = torch.tensor([1., 2., 3., 4., 5.], device=device)

# Reshape c1 to (batch_size, in_channels, length) for conv1d
c1_input = c1.view(1, 1, -1)

kernel = torch.tensor([-1.0, 0.0, 1.0], device=device)
# Reshape kernel to (out_channels, in_channels, kernel_width)
kernel_filter = kernel.view(1, 1, 3)

df = F.conv1d(c1_input, kernel_filter, padding=1)
print(df)

c1_padded = F.pad(c1, (1, 1), mode='constant', value=0)
c1_padded[0] = c1[0]
c1_padded[-1] = c1[-1]

df = F.conv1d(c1_padded.view(1, 1, -1), kernel_filter, padding=0)
print(df)

tensor([[[ 2.,  2.,  2.,  2., -4.]]], device='cuda:0')
tensor([[[1., 2., 2., 2., 1.]]], device='cuda:0')


# sum(a*b)

In [ ]:
import torch

# 1. Setup the Device (Check for NVIDIA GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Initialize Tensors directly on the device
n = 1_000_000
# Tensors are the fundamental unit of storage in PyTorch
a = torch.ones(n, device=device)
b = torch.full((n,), 2.0, device=device)

# 3. Define the operation
def dot_product(x, y):
    # This performs element-wise multiplication followed by a sum
    return torch.sum(x * y)

# 4. (Optional) Compile for speed (similar to JAX JIT)
# In PyTorch 2.0+, this fuses the kernels for faster execution
fast_dot = torch.compile(dot_product)


# 5. Run and measure
# If on GPU, this uses highly optimized CUDA kernels
result = fast_dot(a, b)

result_normal = dot_product(a,b)

print(f"PyTorch Result: {result.item(), " ", result_normal.item()}")

Using device: cuda
PyTorch Result: (2000000.0, ' ', 2000000.0)


# y=Ax

In [ ]:
import torch

# 1. Setup Device (Move to GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Initialize Tensors
# 'requires_grad=True' tells PyTorch to track math for derivatives
A = torch.randn(5000, 5000, device=device)
x = torch.randn(5000, device=device, requires_grad=True)

# 3. Perform Operation
# Standard @ operator works just like NumPy
y = A @ x

optimized_matmul = torch.compile(lambda a, x: a @ x, mode="reduce-overhead")

y1 = optimized_matmul(A, x)

# N particle

In [ ]:
import torch

# 1. Physics Constants
G = 1.0
SOFTENING = 0.1  # Prevents division by zero when particles overlap

def compute_forces(pos, masses):
    """
    Computes gravitational forces for N particles.
    pos: (N, 3) Tensor
    masses: (N, 1) Tensor
    """
    # Create N x N x 3 matrix of relative displacement vectors (pos_j - pos_i)
    # Using [None, :, :] and [:, None, :] triggers broadcasting
    diff = pos[None, :, :] - pos[:, None, :]  # Shape: (N, N, 3)

    # Compute Euclidean distance squared: r^2 + epsilon^2
    dist_sq = torch.sum(diff**2, dim=-1) + SOFTENING**2  # Shape: (N, N)

    # Compute 1 / r^3
    inv_dist_cubed = torch.pow(dist_sq, -1.5)  # Shape: (N, N)

    # Force magnitude: G * m_i * m_j / r^2 -> we multiply by m_j and the vector dr
    # We ignore m_i here because we usually want Force/m_i (Acceleration)
    # Shape: (N, N, 1) * (N, N, 1) * (N, N, 3)
    force_matrix = G * masses[None, :, :] * inv_dist_cubed[:, :, None] * diff

    # Sum forces from all j particles acting on each i particle
    return torch.sum(force_matrix, dim=1)  # Shape: (N, 3)

# --- Simulation Setup ---
N = 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize positions and masses on the GPU
pos = torch.randn(N, 3, device=device)
masses = torch.rand(N, 1, device=device)

# Optional: Compile the function for a 15-20% speedup
fast_forces = torch.compile(compute_forces)

forces = fast_forces(pos, masses)
print(f"Computed forces for {N} bodies on {device}. Shape: {forces.shape}")

Computed forces for 1000 bodies on cuda. Shape: torch.Size([1000, 3])


In [ ]:
# 1. Install PyKeOps
!pip install pykeops

# 2. Re-import to ensure it's loaded properly
import pykeops
import torch

# 3. Perform a health check (Optional but highly recommended)
# This will compile a simple kernel to verify that G++ and CUDA are working.
pykeops.test_torch_bindings()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 30.3 MB/s eta 0:00:00
  Created wheel for pykeops: filename=pykeops-2.3-py3-none-any.whl size=120491 sha256=da47818ca2f0703d708985dffa211ccb031d50635d7ade5afebc968996f88113
  Stored in directory: /root/.cache/pip/wheels/b0/ac/34/a732caf88f9754536a4e2389a3f08c035ad33c0ae6f4cfbc38
  Created wheel for keopscore: filename=keopscore-2.3-py3-none-any.whl size=185897 sha256=88cca494d1c2da66993c929cdabb259d3e88b440a290ad0165d8e83d989ed1b7
  Stored in directory: /root/.cache/pip/wheels/4f/f8/36/9335901e1bdbd1b6c466d8f33800f66f09c60e457ebc921d29
Successfully built pykeops keopscore
[KeOps] Compiling cuda jit compiler engine ... OK
[pyKeOps] Compiling nvrtc binder for python ... OK
[KeOps] Generatin

In [ ]:
import torch
from pykeops.torch import LazyTensor

# 1. Physics Constants
G = 1.0
SOFTENING = 0.01

def compute_forces_keops(pos, masses):
    """
    Computes gravitational forces using Symbolic Tiling.
    Memory usage: O(N)
    Speed: O(N^2) at CUDA speeds
    """
    # Create 'LazyTensors' from our standard PyTorch tensors
    # pos is (N, 3) -> x_i is (N, 1, 3)
    # pos is (N, 3) -> x_j is (1, N, 3)
    x_i = LazyTensor(pos[:, None, :])
    x_j = LazyTensor(pos[None, :, :])

    # masses is (N, 1) -> m_j is (1, N, 1)
    m_j = LazyTensor(masses[None, :, :])

    # 2. Define the 'Symbolic' math
    # No memory is allocated here! This is just a formula.
    diff = x_j - x_i
    dist_sq = (diff**2).sum(-1) + SOFTENING**2

    # Force kernel: G * m_j * (x_j - x_i) / dist^3
    # We use .sum(dim=1) to reduce over the 'j' dimension
    res = (m_j * diff / (dist_sq**1.5)).sum(dim=1)

    return G * res

# --- Simulation Setup ---
N = 100_000  # KeOps can handle N=1,000,000+ on a single GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize positions and masses
pos = torch.randn(N, 3, device=device)
masses = torch.rand(N, 1, device=device)

print(f"Starting KeOps force computation for {N} bodies...")

# Run computation
# The first run will trigger a transparent C++ compilation
forces = compute_forces_keops(pos, masses)

print(f"Computation Complete! Result shape: {forces.shape}")

Starting KeOps force computation for 100000 bodies...
[KeOps] Generating code for Sum_Reduction reduction (with parameters 0) of formula (a*(b-c))/Powf(Sum((b-c)**2)+d,e) with a=Var(0,1,1), b=Var(1,3,1), c=Var(2,3,0), d=Var(3,1,2), e=Var(4,1,2) ... OK
Computation Complete! Result shape: torch.Size([100000, 3])


# with Tile..

In [ ]:
import torch
# 1. Physics Constants
G = 1.0
SOFTENING = 0.01  # Softening parameter for numerical stability

def compute_forces_tiled(pos, masses, tile_size=1024):
    """
    Computes forces for N particles by processing them in manageable tiles.
    This keeps memory usage O(N * tile_size) instead of O(N^2).
    """
    N = pos.shape[0]
    device = pos.device
    forces = torch.zeros_like(pos)

    # Outer loop: Iterate over 'Target' particles in tiles
    for i in range(0, N, tile_size):
        end_i = min(i + tile_size, N)

        # Current batch of particles we are calculating force FOR
        pos_i = pos[i:end_i, None, :]  # Shape: (tile, 1, 3)

        # Inner loop (optional optimization):
        # In this simple version, we broadcast pos_i against ALL particles.
        # This is already memory efficient because 'tile' is small.

        # 2. Compute relative distances (Broadcasting)
        # (tile, 1, 3) - (1, N, 3) -> (tile, N, 3)
        diff = pos[None, :, :] - pos_i

        # 3. Physics Kernel
        dist_sq = torch.sum(diff**2, dim=-1) + SOFTENING**2
        inv_dist_cubed = torch.pow(dist_sq, -1.5)

        # Force/Acceleration components
        # (tile, N, 1) * (1, N, 1) * (tile, N, 3)
        f_tile = G * masses[None, :, :] * inv_dist_cubed[:, :, None] * diff

        # 4. Reduction: Sum all interactions for this tile
        forces[i:end_i] = torch.sum(f_tile, dim=1)

    return forces

# --- Execution ---
N = 10000  # Large enough to crash a naive O(N^2) script on some GPUs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Random initialization
pos = torch.randn(N, 3, device=device)
masses = torch.rand(N, 1, device=device)

# Use torch.compile to fuse the kernel operations for extra speed
fast_forces = torch.compile(compute_forces_tiled)

print(f"Starting tiled force computation for {N} bodies...")
forces = fast_forces(pos, masses, tile_size=1024)
print("Computation Complete!")

Starting tiled force computation for 10000 bodies...
Computation Complete!


# 1D derivative

In [ ]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def pytorch_central_diff(f, dx):
    # 1. Define the Central Difference Kernel: [-1, 0, 1] / (2*dx)
    # The shape must be (out_channels, in_channels, kernel_width)
    kernel = torch.tensor([-0.5, 0.0, 0.5], device=device).view(1, 1, 3) / dx

    # 2. Prepare the input: (batch, channels, length)
    f_tensor = f.view(1, 1, -1)

    # 3. Compute Interior using Convolution (no padding)
    # This results in a tensor of size (N - 2)
    df_interior = F.conv1d(f_tensor, kernel, padding=0)

    # 4. Initialize the result array
    df = torch.zeros_like(f)

    # Assign Interior
    df[1:-1] = df_interior.view(-1)

    # 5. Handle Boundary Conditions
    # Left Point (Forward): (f[1] - f[0]) / dx
    df[0] = (f[1] - f[0]) / dx

    # Right Point (Backward): (f[n-1] - f[n-2]) / dx
    df[-1] = (f[-1] - f[-2]) / dx

    return df

# --- Execution ---
N = 2**16
dx = 1.0 / N
x = torch.linspace(0, 1, N, device=device)
f = torch.sin(2 * torch.pi * x)

df = pytorch_central_diff(f, dx)
print(f"Derivative at index 500: {df[500].item()}")

Derivative at index 500: 6.27587890625


# Derivative for (u,v)

In [ ]:
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Initialize two fields (u and v) with 8 points each
x = torch.linspace(0, 7, 8, device=device)
u = 10.0 * x                # Linear: [0, 10, 20, 30, 40, 50, 60, 70]
v = x**2                   # Quadratic: [0, 1, 4, 9, 16, 25, 36, 49]
dx = 1.0

# 2. Stack them into a Multi-Channel Tensor: (Batch=1, Channels=2, N=8)
# We use 'stack' to put u in Channel 0 and v in Channel 1
uv_tensor = torch.stack([u, v], dim=0).view(1, 2, 8)

# 3. Define the Kernel for 2 Channels
# Shape must be (Out_Channels=2, In_Channels=1, Width=3)
# We use 'repeat' to use the same stencil for both u and v
stencil = torch.tensor([-0.5, 0.0, 0.5], device=device) / dx
kernel = stencil.view(1, 1, 3).repeat(2, 1, 1)

# 4. Perform Multi-Channel Convolution
# PyTorch uses 'groups=2' to ensure Channel 0 uses Kernel 0, and Channel 1 uses Kernel 1
df_interior = F.conv1d(uv_tensor, kernel, padding=0, groups=2)

# 5. Extract Results
# df_interior will have shape (1, 2, 6)
du_interior = df_interior[0, 0, :]
dv_interior = df_interior[0, 1, :]

print(f"u-derivative (Interior): {du_interior.cpu().numpy()}")
print(f"v-derivative (Interior): {dv_interior.cpu().numpy()}")

u-derivative (Interior): [10. 10. 10. 10. 10. 10.]
v-derivative (Interior): [ 2.  4.  6.  8. 10. 12.]


# Compile option

In [ ]:
import torch
import torch.nn.functional as F

# 1. Define your standard PyTorch function
def central_diff(f, dx):
    kernel = torch.tensor([-0.5, 0.0, 0.5], device=f.device).view(1, 1, 3) / dx
    f_tensor = f.view(1, 1, -1)

    # Interior math
    df_interior = F.conv1d(f_tensor, kernel, padding=0)

    # Boundary logic
    df = torch.zeros_like(f)
    df[1:-1] = df_interior.view(-1)
    df[0] = (f[1] - f[0]) / dx
    df[-1] = (f[-1] - f[-2]) / dx
    return df

# 2. THE COMPILATION STEP
# 'mode="reduce-overhead"' is ideal for small-to-medium physics grids
optimized_diff = torch.compile(central_diff, mode="reduce-overhead")

# --- Execution ---
N = 2**16
dx = 1.0 / N
f = torch.sin(torch.linspace(0, 1, N, device="cuda"))

# First call: Triggers compilation (takes a few seconds)
df_first = optimized_diff(f, dx)

# Subsequent calls: Runs at near-native CUDA speed
for _ in range(100):
    df = optimized_diff(f, dx)

W0302 12:04:36.218000 203 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


# Shifted array

In [ ]:
import torch

# 1. Create a 10x10 mock array A
# We'll use a sequence so we can easily see the shift
A = torch.arange(100).view(10, 10)

# 2. Apply the Block Shift with Roll-over
# To move (2, 3) to (0, 0), we shift by -2 and -3
# shifts=(-2, -3) means:
#   - Move up by 2
#   - Move left by 3
B = torch.roll(A, shifts=(-2, -3), dims=(0, 1))

# --- Verification ---
print(f"Original value at A[2, 3]: {A[2, 3].item()}")
print(f"Shifted value at B[0, 0]:  {B[0, 0].item()}")

# Check the roll-over (Periodic Boundary)
# The old B[0, 0] (which was A[0, 0]) should now be at B[8, 7]
print(f"Value that rolled over to B[8, 7]: {B[8, 7].item()}")
print(A)
print(B)

Original value at A[2, 3]: 23
Shifted value at B[0, 0]:  23
Value that rolled over to B[8, 7]: 0
tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
        [20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
        [30, 31, 32, 33, 34, 35, 36, 37, 38, 39],
        [40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
        [50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
        [60, 61, 62, 63, 64, 65, 66, 67, 68, 69],
        [70, 71, 72, 73, 74, 75, 76, 77, 78, 79],
        [80, 81, 82, 83, 84, 85, 86, 87, 88, 89],
        [90, 91, 92, 93, 94, 95, 96, 97, 98, 99]])
tensor([[23, 24, 25, 26, 27, 28, 29, 20, 21, 22],
        [33, 34, 35, 36, 37, 38, 39, 30, 31, 32],
        [43, 44, 45, 46, 47, 48, 49, 40, 41, 42],
        [53, 54, 55, 56, 57, 58, 59, 50, 51, 52],
        [63, 64, 65, 66, 67, 68, 69, 60, 61, 62],
        [73, 74, 75, 76, 77, 78, 79, 70, 71, 72],
        [83, 84, 85, 86, 87, 88, 89, 80, 81, 82],
        [93, 94, 95, 96, 97, 98, 99, 90, 91, 92],
  